# Stitch Integration

This notebook demonstrates the **Stitch** stage of the Foreign Whispers dubbing pipeline.
It performs final video assembly: combining the original video with dubbed TTS audio and
rolling two-line translated captions via ffmpeg. The stitch uses audio-only remux
(no re-encoding), preserving original video quality.

**Prerequisites:**
- The Docker stack must be running (`docker compose --profile nvidia up -d`).
- The API should be accessible at `http://localhost:8080`.
- Prior pipeline stages (download, transcribe, translate, TTS) must have completed for the target video.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE, ALIGNED

fw = FWClient("http://localhost:8080")
fw.healthz()

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-stitch")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

Project root: /home/ec2-user/foreign-whispers
Logfire not configured — using no-op shim.


In [2]:
#It checks that the required original video, 
# TTS WAV, and caption/translation inputs exist before calling stitch.
from pathlib import Path

video_id = "GYQ5yGV_-Oc"
CONFIG = ALIGNED

# Resolve likely title from video registry/API output if needed
videos = fw.videos()
video = next((v for v in videos if v["id"] == video_id), None)

if video is None:
    raise ValueError(f"Video {video_id} not found in API registry")

title = video["title"]

original_video = PROJECT_ROOT / "pipeline_data" / "api" / "videos" / f"{title}.mp4"
tts_wav = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox" / CONFIG / f"{title}.wav"
translation_json = PROJECT_ROOT / "pipeline_data" / "api" / "translations" / "argos" / f"{title}.json"

print("Video title:", title)
print("Config:", CONFIG)
print("Original video exists:", original_video.exists(), original_video)
print("TTS WAV exists:", tts_wav.exists(), tts_wav)
print("Translation JSON exists:", translation_json.exists(), translation_json)

Video title: Strait of Hormuz disruption threatens to shake global economy
Config: c-86ab861
Original video exists: True /home/ec2-user/foreign-whispers/pipeline_data/api/videos/Strait of Hormuz disruption threatens to shake global economy.mp4
TTS WAV exists: True /home/ec2-user/foreign-whispers/pipeline_data/api/tts_audio/chatterbox/c-86ab861/Strait of Hormuz disruption threatens to shake global economy.wav
Translation JSON exists: True /home/ec2-user/foreign-whispers/pipeline_data/api/translations/argos/Strait of Hormuz disruption threatens to shake global economy.json


## Stitch Video

Call the API to combine the original video with the dubbed TTS audio track.
The stitch endpoint replaces the audio stream via ffmpeg remux (no video re-encoding)
and generates rolling two-line VTT captions.

In [3]:
video_id = "GYQ5yGV_-Oc"

with logfire.span("stitch", video_id=video_id, config=ALIGNED):
    result = fw.stitch(video_id, config=ALIGNED)

print(f"Video ID:   {result['video_id']}")
print(f"Video path: {result['video_path']}")
print(f"Config:     {result['config']}")

Video ID:   GYQ5yGV_-Oc
Video path: /app/pipeline_data/api/dubbed_videos/c-86ab861/Strait of Hormuz disruption threatens to shake global economy.mp4
Config:     c-86ab861


## Inspect Output Artifacts

The stitch stage produces files in two directories:

- `pipeline_data/api/dubbed_videos/{config}/` — final dubbed MP4 files
- `pipeline_data/api/dubbed_captions/` — target-language VTT caption files

In [4]:
from pathlib import Path

CONFIG = ALIGNED  # or BASELINE

dubbed_videos_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_videos" / CONFIG
dubbed_captions_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_captions"

print(f"Config: {CONFIG}")

print("\nDubbed videos:")
if dubbed_videos_dir.exists():
    for f in sorted(dubbed_videos_dir.glob("*.mp4")):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name}  ({size_mb:.1f} MB)")
else:
    print(f"  No folder found: {dubbed_videos_dir}")

print("\nDubbed captions:")
if dubbed_captions_dir.exists():
    for f in sorted(dubbed_captions_dir.glob("*.vtt")):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name}  ({size_kb:.1f} KB)")
else:
    print(f"  No folder found: {dubbed_captions_dir}")

Config: c-86ab861

Dubbed videos:
  Strait of Hormuz disruption threatens to shake global economy.mp4  (62.1 MB)

Dubbed captions:
  Strait of Hormuz disruption threatens to shake global economy.vtt  (123.4 KB)


## View Generated Captions

The stitch stage generates VTT captions in a rolling two-line format:
the current translated line appears on top, and the previous line is shown
on the bottom, giving viewers context continuity.

In [5]:
# Find the VTT file for this video
vtt_files = list(dubbed_captions_dir.rglob(f"{video_id}*.vtt"))
if not vtt_files:
    vtt_files = list(dubbed_captions_dir.rglob("*.vtt"))

if vtt_files:
    vtt_path = vtt_files[0]
    print(f"Caption file: {vtt_path.name}\n")
    lines = vtt_path.read_text().splitlines()
    for line in lines[:30]:
        print(line)
    if len(lines) > 30:
        print(f"\n... ({len(lines) - 30} more lines)")
    print()
    print("Note the rolling two-line pattern: each cue shows the current")
    print("translated line on top and the previous line on the bottom,")
    print("providing continuity for the viewer.")
else:
    print("No VTT files found. Run the stitch step first.")

Caption file: Strait of Hormuz disruption threatens to shake global economy.vtt

WEBVTT

1
00:00:07.990 --> 00:00:08.000
¿Cuál es el peor escenario del caso?

2
00:00:08.000 --> 00:00:10.230
¿Cuál es el peor escenario del caso que estás haciendo?00:00:08.240 Cursoc Emperado preocupado implicado 00:00:08.480 Fuertec saber acerca de lo que hiciste/c Empezar Nocturas 00:00:09.360 Cómodo es escrito/c Ingreso garantizado00:00:09.599 Ingreso incluido que fue nombrado/c Ingresado00:00:09.760 Ingreso/c Fuerte garantizado00:00:10.000c
¿Cuál es el peor escenario del caso?

3
00:00:10.230 --> 00:00:10.240
Te preocupa que sea
¿Cuál es el peor escenario del caso que estás haciendo?00:00:08.240 Cursoc Emperado preocupado implicado 00:00:08.480 Fuertec saber acerca de lo que hiciste/c Empezar Nocturas 00:00:09.360 Cómodo es escrito/c Ingreso garantizado00:00:09.599 Ingreso incluido que fue nombrado/c Ingresado00:00:09.760 Ingreso/c Fuerte garantizado00:00:10.000c

4
00:00:10.240 --> 00:00:12.390
te p

In [6]:
import subprocess
import json
from pathlib import Path

raw_output_path = Path(result["video_path"])

# API returns container paths like /app/pipeline_data/...
# Convert them to host paths under PROJECT_ROOT.
if str(raw_output_path).startswith("/app/"):
    output_video_path = PROJECT_ROOT / str(raw_output_path).removeprefix("/app/")
elif raw_output_path.is_absolute():
    output_video_path = raw_output_path
else:
    output_video_path = PROJECT_ROOT / raw_output_path

print("Raw API path:", result["video_path"])
print("Host output path:", output_video_path)
print("Output exists on host:", output_video_path.exists())

Raw API path: /app/pipeline_data/api/dubbed_videos/c-86ab861/Strait of Hormuz disruption threatens to shake global economy.mp4
Host output path: /home/ec2-user/foreign-whispers/pipeline_data/api/dubbed_videos/c-86ab861/Strait of Hormuz disruption threatens to shake global economy.mp4
Output exists on host: True


## Playback

To play the dubbed output:

1. **Frontend:** Open <http://localhost:8501> and select the video from the list.
   The UI will load the dubbed MP4 with captions overlay.

2. **Direct file:** Play the MP4 directly from
   `pipeline_data/api/dubbed_videos/{config}/{video_id}.mp4` using any media player
   (e.g., VLC, mpv). Load the corresponding VTT file from `pipeline_data/api/dubbed_captions/`
   as an external subtitle track.

## Summary

The stitch stage produced:

- **Dubbed MP4** in `pipeline_data/api/dubbed_videos/{config}/` — original video with replaced audio track
- **VTT captions** in `pipeline_data/api/dubbed_captions/` — rolling two-line translated subtitles

Audio-only remux means no quality loss on the video track: ffmpeg copies the video
stream as-is and only replaces the audio stream with the synthesized TTS output.